<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Conociendo la Finca Antes de Sembrar: Mirar los Datos Antes de Calcular 🥔
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 05
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/05%20-%20Exploracion%20y%20Limpieza%20de%20Datos/Para%20Dummies/00_EDA_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del **cuaderno estándar**. Antes de comprar semilla, un buen agricultor **camina la finca**: mira el terreno, cuenta los surcos y revisa si hay algo raro. Con los datos pasa igual: **antes de calcular nada, hay que mirarlos**. Aprenderás tres hábitos:

| Hábito | En la finca | En los datos |
|---|---|---|
| **El vistazo** | Contar cuántas matas hay, cuáles faltan, la más grande y la más chica | Cuántos datos hay, cuántos faltan, mínimo, máximo, promedio y mediana |
| **Cambiar la escala** | Medir en «veces más grande» cuando una finca es gigante | Usar el **logaritmo** para que lo enorme no aplaste lo pequeño |
| **Desconfiar de la casualidad** | Si buscas mucho, siempre encuentras «algo especial» | Mirar muchas relaciones produce **falsos hallazgos** por puro azar |

> 🔗 Si luego quieres ver todo **con rigor y desde cero**: **cuaderno estándar**. Después de esto: **El cuaderno con páginas rotas**.

---
## 1. El Vistazo: Doce Fincas y un Número Raro 👀

Anotamos el rendimiento de **12 fincas** (toneladas por hectárea; datos **inventados**). A una finca **no le anotaron** el dato y a otra le anotaron **21.0**, que se ve muy distinto de las demás. Antes de promediar, hacemos el vistazo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rend = pd.Series([6.2, 7.1, 5.8, 6.9, 7.4, 6.5, np.nan, 5.9, 7.0, 6.6, 6.8, 21.0], name="rendimiento")


def vistazo(s):
    """Lo mínimo que hay que mirar de una columna: cuántos datos, cuántos faltan, extremos, promedio y mediana."""
    return {"cuántas": int(s.notna().sum()), "faltan": int(s.isna().sum()), "mínimo": s.min(), "máximo": s.max(),
            "promedio": round(s.mean(), 2), "mediana": s.median()}


v = vistazo(rend)
print(v)
assert v["faltan"] == 1 and v["cuántas"] == 11 and v["máximo"] == 21.0
assert v["promedio"] > v["mediana"] + 0.5                                   # ese 21.0 jala el promedio hacia arriba

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, 13), rend.fillna(0), color=["#dc2626" if x > 15 else "#2563eb" for x in rend.fillna(0)])
ax.axhline(v["promedio"], color="#f59e0b", ls="--", label=f"promedio {v['promedio']}")
ax.axhline(v["mediana"], color="#16a34a", ls="--", label=f"mediana {v['mediana']}")
ax.set_xlabel("finca (la 7 no tiene dato)"); ax.set_ylabel("t/ha"); ax.legend(); ax.set_title("El vistazo: una finca se sale del patrón", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()

---
**Lección:** el vistazo ya contó **dos historias**: (1) **falta un dato** y (2) hay un **21.0** que arrastra el promedio a 7.93 mientras la mediana sigue en 6.8. ¿Es un error de digitación (¿era 2.1?), o una finca extraordinaria? **No lo borres ni lo «arregles» sin preguntar**: un dato raro es una **pregunta**, no un error asegurado.

---

## 2. Cuando una Finca es Gigante: Cambiar de Escala 🔭

Estas son las áreas de 10 fincas, en hectáreas (**inventadas**): nueve son pequeñas y una mide 60. En una gráfica normal, la gigante **aplasta a todas las demás**. El **logaritmo** convierte «veces más grande» en distancias parejas: duplicar el área suma siempre lo mismo.

In [ ]:
areas = np.array([0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0, 4.0, 60.0])

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].bar(range(10), areas, color="#2563eb"); ax[0].set_title("Área en hectáreas: la última aplasta a las demás", fontsize=9)
ax[1].bar(range(10), np.log10(areas) - np.log10(areas).min(), color="#16a34a"); ax[1].set_title("En escala logarítmica se ven todas", fontsize=9)
for a in ax:
    a.set_xlabel("finca")
plt.tight_layout(); plt.show()

veces = areas.max() / areas.min()
print(f"La finca más grande es {veces:.0f} veces la más pequeña; en logaritmo (base 10) la diferencia es solo {np.log10(veces):.2f}.")
assert np.isclose(veces, 120) and np.isclose(np.log10(veces), 2.0792, atol=1e-3)
assert np.isclose(np.log10(2.0) - np.log10(1.0), np.log10(4.0) - np.log10(2.0))        # duplicar suma siempre lo mismo

---
**Lección:** la finca de 60 ha es **120 veces** la de 0.5 ha, pero en logaritmo la diferencia es solo **2.08**, comparable con las demás. Por eso, cuando un dato crece **por multiplicación** (áreas, ingresos, precios), conviene mirar su **logaritmo**.

---

## 3. Si Buscas Mucho, Algo «Especial» Aparece 🪙

200 personas lanzan una moneda **5 veces cada una**. Alguien sacará **5 caras seguidas** por pura suerte: con 200 intentos se esperan unas **6** (200 ÷ 32). Si solo miraras a esas personas dirías «¡qué buena mano!». Con los datos pasa lo mismo: **si comparas cientos de relaciones, algunas parecerán importantes solo por casualidad**.

In [ ]:
rng = np.random.default_rng(3)
personas = 200
caras = rng.integers(0, 2, size=(personas, 5))                    # 1 = cara; cada fila = una persona y sus 5 lanzamientos
todas_caras = int((caras.sum(axis=1) == 5).sum())
print(f"{todas_caras} de {personas} personas sacaron 5 caras seguidas (se esperaban ≈ {personas / 32:.1f}), sin tener ninguna 'buena mano'.")
assert 2 <= todas_caras <= 12

---
##### 🛠️ Práctica 1: ¿Cuáles datos son sospechosos?

Con `rend` (los rendimientos de las 12 fincas):

1. Calcula `mediana` (con `rend.median()`).
2. Guarda en `sospechosos` los datos **mayores que 3 veces la mediana** (pista: `rend[rend > 3 * mediana]`).
3. Comprueba con `assert` que hay **un solo** sospechoso y que vale 21.0. ¿Lo borrarías? ¿Qué le preguntarías al dueño de la finca?

In [ ]:
# Escribe tu código aquí

# mediana = ...
# sospechosos = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
mediana = rend.median()
sospechosos = rend[rend > 3 * mediana]
print("Mediana:", mediana, "| sospechosos:", sospechosos.to_dict())

assert len(sospechosos) == 1 and sospechosos.iloc[0] == 21.0
print("No se borra a la ligera: se pregunta si fue un error de digitación (¿2.1?) o una finca realmente excepcional.")
```
</details>

---


---
## Resumen en una frase 🎯

> **Mira los datos antes de calcular: cuenta cuántos hay, busca lo raro, cambia de escala cuando algo es enorme y desconfía de los «hallazgos» que aparecen al buscar mucho.**

### Lo que aprendiste hoy:
- ✅ El **vistazo** (cuántos datos, cuántos faltan, extremos, promedio y mediana) detecta problemas **antes** de cualquier análisis.
- ✅ Un solo **21.0** subió el promedio a 7.93 mientras la mediana quedó en 6.8; un dato raro es una **pregunta**, no un error asegurado.
- ✅ El **logaritmo** convierte «120 veces más grande» en una diferencia de 2.08: sirve cuando algo crece por multiplicación.
- ✅ Si buscas mucho, hay **casualidades**: de 200 personas, varias sacan 5 caras seguidas sin ser especiales.

> 🎓 **Siguiente paso:** el **cuaderno estándar** hace todo esto con rigor (intervalos, comparaciones múltiples, transformaciones). Para seguir: **El cuaderno con páginas rotas**.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>